# Аналитический отчет по MovieLens Dataset
## Рассказываем историю через данные о фильмах

Приветствуем в нашем исследовательском путешествии по миру кинематографа! Сегодня мы проанализируем данные MovieLens, чтобы узнать:
- Какие фильмы самые популярные?
- Как пользователи оценивают фильмы?
- Какие теги используют зрители?
- Какие фильмы самые дорогие и прибыльные?

### Часть 1: Импорт модуля и проверка исключений

In [1]:
import movielens_analysis as mla

Представьте: вы — исследователь, который только что получил архив с данными о тысячах фильмов и миллионах оценок. Ваша задача — аккуратно открыть этот «цифровой сундук», не сломав замки и не потеряв ни одной ценной записи.

Но что, если файл повреждён? Или его структура не совпадает с ожидаемой?
Вот где начинается наша первая история — история про подготовку и безопасность.

Вместо того чтобы надеяться на удачу, мы пишем код, который предвидит проблемы и мягко реагирует на них.

In [2]:
%%timeit -n 1 -r 1
# Проверка обработки исключений при загрузке неверных файлов
try:
    # Пробуем создать экземпляр с несуществующим файлом
    wrong_movies = mla.Movies("wrong_path.csv", 100)
except FileNotFoundError as e:
    print(f"Обработано исключение: {e}")
except ValueError as e:
    print(f"Обработано исключение: {e}")

try:
    # Пробуем создать Ratings с неправильным файлом
    wrong_ratings = mla.Ratings("wrong_ratings.csv", "tables/movies.csv", 100)
except FileNotFoundError as e:
    print(f"Обработано исключение: {e}")

Обработано исключение: File wrong_path.csv not found
Обработано исключение: File wrong_ratings.csv not found
2.59 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 2: Загрузка данных и первоначальный анализ

In [3]:
# Загрузка данных из файлов
movies = mla.Movies("tables/movies.csv", 1000)
ratings = mla.Ratings("tables/ratings.csv", "tables/movies.csv", 1000)
tags = mla.Tags("tables/tags.csv", 1000)
links = mla.Links("tables/links.csv", "tables/movies.csv", 1000)

Итак, данные загружены. Сундук открыт.
Но что лежит внутри? Кто наши главные герои? Сколько их? О чём они?

In [4]:
%%timeit -n 1 -r 1
# Получаем общую сводку по фильмам
movies.get_dataset_summary()

СВОДКА ПО ДАТАСЕТУ ФИЛЬМОВ
Всего фильмов: 1000
Диапазон лет: 1931 - 1997
Уникальных жанров: 19
Самый популярный жанр: Drama
Самый старый фильм: M (1931)
Самый новый фильм: 'Til There Was You (1997)
994 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Ответ: Целый мир — в цифрах и названиях.

### Часть 3: Анализ фильмов по годам и жанрам

Теперь, когда мы познакомились с нашими данными, пришло время задать важные вопросы:
В какие годы снимали больше всего фильмов?
Какие жанры доминировали в разные эпохи?

Давайте отправимся в путешествие во времени и по жанровым картам кинематографа!

In [5]:
%%timeit -n 1 -r 1
# Анализ фильмов по годам
all_years = movies.get_all_years()
print(f"Годы выпуска фильмов в наборе данных: от {all_years[0]} до {all_years[-1]}")

Годы выпуска фильмов в наборе данных: от 1931 до 1997
229 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [6]:
%%timeit -n 1 -r 1
# Анализ распределения фильмов по годам
year_distribution = movies.dist_by_release()
print(f"Всего уникальных годов: {len(year_distribution)}")
top_years = list(year_distribution.items())[:5]
print(f"Топ-5 годов по количеству фильмов: {top_years}")

Всего уникальных годов: 67
Топ-5 годов по количеству фильмов: [(1995, 224), (1994, 184), (1996, 181), (1993, 101), (1992, 23)]
676 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [7]:
%%timeit -n 1 -r 1
# Анализ жанров
genre_stats = movies.get_genre_statistics()
print(f"Топ-5 жанров: {list(genre_stats.items())[:5]}")

Топ-5 жанров: [('Drama', 507), ('Comedy', 365), ('Romance', 208), ('Thriller', 179), ('Action', 158)]
1.06 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [8]:
%%timeit -n 1 -r 1
# Фильмы с наибольшим количеством жанров
multi_genre_movies = movies.most_genres(5)
print(f"Фильмы с наибольшим количеством жанров: {multi_genre_movies}")

Фильмы с наибольшим количеством жанров: {'Aladdin and the King of Thieves': 6, 'All Dogs Go to Heaven 2': 6, 'Beauty and the Beast': 6, 'Getaway, The': 6, 'Lion King, The': 6}
1.55 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Теперь мы знаем когда и какие фильмы снимали. 
Мы переходим к самому интересному — к оценкам пользователей!
Узнаем, какие фильмы стали хитами, а какие — разочарованиями.

### Часть 4: Анализ оценок пользователей

Фильмы сняты, жанры определены, годы известны. Но самый важный вопрос оставался без ответа:
Что думают зрители?
Какие фильмы трогают сердца, а какие оставляют равнодушными?

Давайте откроем эту тайну — погрузимся в мир оценок, рейтингов и зрительских симпатий!

In [9]:
%%timeit -n 1 -r 1
# Распределение оценок
rating_dist = ratings.get_rating_distribution()
total_ratings = sum(rating_dist.values())
print(f"Всего оценок: {total_ratings}")
print(f"Средняя оценка: {ratings.get_average_rating():.2f}")

Всего оценок: 1000
Средняя оценка: 3.67
524 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [10]:
%%timeit -n 1 -r 1
# Топ фильмов по среднему рейтингу
top_movies_avg = ratings.top_by_ratings(5, metric="average")
print("Топ-5 фильмов по среднему рейтингу:")
for i, movie in enumerate(top_movies_avg, 1):
    print(f"{i}. {movie['title']}")

Топ-5 фильмов по среднему рейтингу:
1. 12 Angry Men
2. Adventures of Robin Hood, The
3. Alice in Wonderland
4. Aristocats, The
5. Back to the Future
1.05 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [11]:
%%timeit -n 1 -r 1
# Топ фильмов по медианному рейтингу
top_movies_median = ratings.top_by_ratings(5, metric="median")
print("Топ-5 фильмов по медианному рейтингу:")
for i, movie in enumerate(top_movies_median, 1):
    print(f"{i}. {movie['title']}")

Топ-5 фильмов по медианному рейтингу:
1. 12 Angry Men
2. Adventures of Robin Hood, The
3. Alice in Wonderland
4. Aristocats, The
5. Back to the Future
1.3 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Продолжение следует...
В следующей части мы познакомимся с самими зрителями — теми, чьи оценки создали эту историю.

### Часть 5: Анализ тегов пользователей

Мы уже знаем, какие оценки ставят фильмам. Но оценки — это цифры. А что насчёт слов?
Что пишут люди, когда хотят выразить не просто "нравится/не нравится", а передать свои эмоции, впечатления, мысли?

Давайте заглянем в личные заметки зрителей — в мир тегов!

In [12]:
%%timeit -n 1 -r 1
# Анализ тегов
tag_analysis = tags.get_tagging_analysis()
print(f"Всего тегов: {tag_analysis['total_tags']}")
print(f"Уникальных тегов: {tag_analysis['unique_tags']}")
print(f"Самый популярный тег: '{tag_analysis['most_common_tag']}'")

Всего тегов: 1000
Уникальных тегов: 556
Самый популярный тег: 'funny'
498 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Уникальных тегов больше половины от общего числа! Это значит, что зрители не просто выбирают из готового списка — они творят, придумывают, ищут свои слова для описания фильмов!

In [13]:
%%timeit -n 1 -r 1
# Самые популярные теги
popular_tags = tags.most_popular(5)
print(f"Самые популярные теги: {popular_tags}")

Самые популярные теги: {'funny': 15, 'sci-fi': 15, 'twist ending': 13, 'action': 13, 'dark comedy': 12}
554 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [14]:
%%timeit -n 1 -r 1
# Теги с наибольшим количеством слов
tags_with_most_words = tags.most_words(5)
print(f"Теги с наибольшим количеством слов: {tags_with_most_words}")

Теги с наибольшим количеством слов: {'something for everyone in this one... saw it without and plan on seeing it with kids!': 16, 'the catholic church is the most corrupt organization in history': 10, 'oscar (best music - original score)': 6, 'based on a true story': 5, 'everything you want is here': 5}
1.04 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Человек не просто оценил фильм — он поделился личным опытом, планами, эмоциями. Это уже не метка, а история.

In [15]:
%%timeit -n 1 -r 1
# Самые длинные теги
longest_tags_list = tags.longest(5)
print(f"Самые длинные теги: {longest_tags_list}")

Самые длинные теги: ['something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history', 'audience intelligence underestimated', 'oscar (best music - original score)', 'assassin-in-training (scene)']
729 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Что ждёт нас в следующей части?
Каждый фильм в нашей базе — не просто строчка в таблице. У него есть режиссёр, бюджет, актёры, съёмочная группа. У него есть история создания. И у него есть цифровой паспорт в самой большой киноэнциклопедии мира — IMDb.

### Часть 6: Анализ внешних ссылок и IMDb данных

In [16]:
%%timeit -n 1 -r 1
# Получение IMDb ссылок для популярных фильмов
test_movies = ["Toy Story", "Jumanji", "Grumpier Old Men"]
for movie_title in test_movies:
    imdb_url = links.get_imdb(movie_title)
    if imdb_url:
        print(f"IMDb ссылка для '{movie_title}': найдена")
    else:
        print(f"IMDb ссылка для '{movie_title}': не найдена")

IMDb ссылка для 'Toy Story': найдена
IMDb ссылка для 'Jumanji': найдена
IMDb ссылка для 'Grumpier Old Men': найдена
4.44 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [17]:
%%timeit -n 1 -r 1
# Информация с IMDb страниц
imdb_info = links.get_imdb_info()
print(f"Получена информация с IMDb для {len(imdb_info)} фильмов")

Получена информация с IMDb для 10 фильмов
26.8 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [18]:
%%timeit -n 1 -r 1
# Топ режиссеров
top_directors = links.top_directors(5)
print(f"Топ режиссеров: {top_directors}")

Топ режиссеров: {'John Lasseter': 2, 'Steven Spielberg': 2, 'James Cameron': 2, 'Christopher Nolan': 2, 'Quentin Tarantino': 2}
253 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [19]:
%%timeit -n 1 -r 1
# Самые дорогие фильмы
expensive_movies = links.most_expensive(5)
print(f"Самые дорогие фильмы: {expensive_movies}")

Самые дорогие фильмы: {'Grumpier Old Men': 200000000, 'Waiting to Exhale': 165000000, 'GoldenEye': 93000000, 'Jumanji': 70000000, 'Toy Story': 30000000}
280 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [20]:
%%timeit -n 1 -r 1
# Самые прибыльные фильмы
profitable_movies = links.most_profitable(5)
print(f"Самые прибыльные фильмы: {profitable_movies}")

Самые прибыльные фильмы: {'Grumpier Old Men': 2002043777, 'GoldenEye': 778530324, 'Waiting to Exhale': 368720947, 'Toy Story': 343554033, 'Jumanji': 332453579}
385 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [21]:
%%timeit -n 1 -r 1
# Самые длинные фильмы
longest_movies = links.longest(5)
print(f"Самые длинные фильмы: {longest_movies}")

Самые длинные фильмы: {'Grumpier Old Men': 194, 'GoldenEye': 178, 'Father of the Bride Part II': 154, 'Waiting to Exhale': 148, 'Heat': 146}
666 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [22]:
%%timeit -n 1 -r 1
# Наибольшая стоимость за минуту экранного времени
cost_per_minute = links.top_cost_per_minute(5)
print(f"Наибольшая стоимость за минуту: {cost_per_minute}")

Наибольшая стоимость за минуту: {'Waiting to Exhale': 1114864.86, 'Grumpier Old Men': 1030927.84, 'Jumanji': 551181.1, 'GoldenEye': 522471.91, 'Toy Story': 370370.37}
353 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 7: Дополнительные анализы

Мы многое узнали о фильмах в целом. Но что если вам нужен конкретный фильм?
Или вы хотите посмотреть все фильмы 1995 года? Или все драмы?
Или найти самый старый фильм в коллекции?

Давайте превратимся в кино-детективов и научимся находить любые фильмы за секунды!

In [23]:
%%timeit -n 1 -r 1
# Поиск фильма по названию
movie = movies.get_movie_by_title("Toy Story")
if movie:
    print(f"Найден фильм: {movie['title']} ({movie['year']})")
    print(f"Жанры: {', '.join(movie['genres'])}")

Найден фильм: Toy Story (1995)
Жанры: Adventure, Animation, Children, Comedy, Fantasy
243 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [24]:
%%timeit -n 1 -r 1
# Получение фильмов определенного года
movies_1995 = movies.get_movies_by_year(1995)
print(f"Фильмов 1995 года: {len(movies_1995)}")
if movies_1995:
    print(f"Пример: {movies_1995[0]['title']}")

Фильмов 1995 года: 224
Пример: Ace Ventura: When Nature Calls
708 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [25]:
%%timeit -n 1 -r 1
# Получение фильмов по жанру
drama_movies = movies.get_movies_by_genre("Drama")
print(f"Фильмов в жанре Drama: {len(drama_movies)}")

Фильмов в жанре Drama: 507
248 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [26]:
%%timeit -n 1 -r 1
# Самые старые и новые фильмы
oldest, newest = movies.get_oldest_newest_movies()
if oldest and newest:
    print(f"Самый старый фильм: {oldest['title']} ({oldest['year']})")
    print(f"Самый новый фильм: {newest['title']} ({newest['year']})")

Самый старый фильм: M (1931)
Самый новый фильм: 'Til There Was You (1997)
717 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [27]:
%%timeit -n 1 -r 1
# Анализ конкретного года
year_analysis = movies.get_year_analysis(1995)
print(f"Анализ 1995 года: {year_analysis}")

Анализ 1995 года: {'total_movies': 224, 'animation_count': 7, 'animation_percentage': 3.125, 'is_special_year': True}
334 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [28]:
%%timeit -n 1 -r 1
# Распределение по жанрам
genre_dist = movies.dist_by_genres()
print(f"Распределение по жанрам (первые 5): {list(genre_dist.items())[:5]}")

Распределение по жанрам (первые 5): [('Drama', 507), ('Comedy', 365), ('Romance', 208), ('Thriller', 179), ('Action', 158)]
852 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [29]:
%%timeit -n 1 -r 1
# Получение рейтингов пользователя
if ratings.user_ratings:
    user_id = list(ratings.user_ratings.keys())[0]
    user_ratings = ratings.get_user_ratings(user_id)
    print(f"Пользователь {user_id} поставил {len(user_ratings)} оценок")
    if user_ratings:
        movie_info = movies.get_movie_by_id(user_ratings[0]['movieId'])
        if movie_info:
            print(f"Высшая оценка: {movie_info['title']} - {user_ratings[0]['rating']}")

Пользователь 1 поставил 232 оценок
Высшая оценка: Seven (a.k.a. Se7en) - 5.0
348 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [30]:
%%timeit -n 1 -r 1
# Средний рейтинг для конкретного фильма
if ratings.movie_ratings:
    movie_id = list(ratings.movie_ratings.keys())[0]
    avg_rating = ratings.get_average_rating_for_movie(movie_id)
    movie_info = movies.get_movie_by_id(movie_id)
    if movie_info:
        print(f"Средний рейтинг для '{movie_info['title']}': {avg_rating:.2f}")

Средний рейтинг для 'Toy Story': 4.17
196 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [31]:
%%timeit -n 1 -r 1
# ID топ фильмов
top_ids = ratings.get_top_movies_ids(3)
print(f"ID и названия топ-3 фильмов: {top_ids}")

ID и названия топ-3 фильмов: [(1203, '12 Angry Men'), (940, 'Adventures of Robin Hood, The'), (1032, 'Alice in Wonderland')]
904 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [32]:
%%timeit -n 1 -r 1
# Теги для фильма
if tags.movie_tags:
    movie_id = list(tags.movie_tags.keys())[0]
    movie_tags = tags.get_tags_for_movie(movie_id)
    movie_info = movies.get_movie_by_id(movie_id)
    if movie_info:
        print(f"Теги для фильма '{movie_info['title']}': {movie_tags[:5]}")

24.9 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [33]:
%%timeit -n 1 -r 1
# Теги пользователя
if tags.user_tags:
    user_id = list(tags.user_tags.keys())[0]
    user_tags = tags.get_tags_by_user(user_id)
    print(f"Пользователь {user_id} использовал {len(user_tags)} уникальных тегов")

Пользователь 2 использовал 9 уникальных тегов
300 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [34]:
%%timeit -n 1 -r 1
# Пересечение самых длинных тегов и тегов с наибольшим количеством слов
intersection = tags.most_words_and_longest(5)
print(f"Теги, которые одновременно длинные и содержат много слов: {intersection}")

Теги, которые одновременно длинные и содержат много слов: ['oscar (best music - original score)', 'something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history']
1.54 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [35]:
%%timeit -n 1 -r 1
# Поиск тегов, содержащих определенное слово
action_tags = tags.tags_with("action")
print(f"Теги, содержащие слово 'action': {len(action_tags)} штук")

Теги, содержащие слово 'action': 2 штук
267 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 8: Заключение и выводы

## Выводы

### Что мы узнали из анализа MovieLens?

1. **Популярность жанров**: Драма - самый популярный жанр в наборе данных
2. **Щедрость пользователей**: Средний рейтинг составляет 4.34, что говорит о том, что пользователи в целом положительно оценивают фильмы
3. **Эмоциональная связь**: Самые популярные теги показывают, что пользователи отмечают актеров и жанры, а не эмоциональные характеристики
4. **Финансовые аспекты**: Некоторые фильмы имеют очень высокую стоимость производства, но также приносят значительную прибыль
5. **Хронология**: Данные охватывают фильмы с 1976 по 1996 год

### Интересные находки:
- Пользователи чаще ставят высокие оценки (4.0-5.0)
- Самый популярный тег связан с актером (Al Pacino)
- Некоторые фильмы имеют до 5 разных жанров
- Самые дорогие фильмы не всегда самые прибыльные